# STEP 5: LLM-Assisted Labeling (Optional but Recommended)

For cases where rule-based labeling returned -1 (unknown), we use Gemini to label them from the case title/description.

This is OPTIONAL. Run this only if:
- You have needs_llm_labeling.parquet from step 04
- You want to expand your training data beyond rule-based labels
- You want to validate the rule-based labels on a sample

Cost estimate:
- ~500 tokens per case (title + description + prompt)
- At Gemini API pricing, 1000 cases ≈ affordable for a project
- Recommend batching: do 500–2000 cases for a good validation set

Output: llm_labeled_sample.parquet

In [25]:
# Setup with Groq
%pip install -q requests pandas pyarrow groq

import pandas as pd
import re
import gc
import requests
import json
import time
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq
from groq import Groq

In [26]:
# Connect Google Drive (Colab)
from google.colab import drive

drive.mount('/content/drive')
print('Connected to the drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Connected to the drive


## Setup Packages and API Configuration

In [ ]:
# Groq API Configuration
import os
from groq import Groq

# Configuration (Drive paths)
BASE_DIR = Path('/content/drive/MyDrive/MiniProject')
OUTPUT_DIR = BASE_DIR / 'compiled_dataset_180426'

# Groq API Keys (Add your keys here)
GROQ_API_KEYS = [
    # Removed API key for security
]

# API Settings
GROQ_MODEL = "llama-3.3-70b-versatile"
current_key_index = 0

def get_groq_client():
    return Groq(api_key=GROQ_API_KEYS[current_key_index])

client = get_groq_client()

print(f"Groq configured with model: {GROQ_MODEL}")

Groq configured with model: llama-3.3-70b-versatile


## Define Labeling Prompt and Functions

In [28]:
# Updated labeling function for Groq
def label_case_with_llm(row: pd.Series) -> dict:
    prompt = LABELING_PROMPT.format(
        court_level=row.get("court_level", "Unknown"),
        case_type=row.get("case_type", "Unknown"),
        act=row.get("act", "Unknown"),
        title=str(row.get("title", ""))[:300],
        description=str(row.get("description", ""))[:500],
        disposal_nature=row.get("disposal_nature", "Unknown"),
    )

    try:
        completion = client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
            temperature=0.1,
        )
        result = json.loads(completion.choices[0].message.content)
        return {
            "adr_label": 1 if result.get("adr_suitable") else 0,
            "odr_label": 1 if result.get("odr_suitable") else 0,
            "adr_target": 1 if result.get("adr_suitable") else 0,
            "odr_target": 1 if result.get("odr_suitable") else 0,
            "final_label": 2 if result.get("odr_suitable") else (1 if result.get("adr_suitable") else 0),
            "label_reason": f"Groq ({result.get('confidence', '?')}): {result.get('reasoning', '')}",
            "llm_confidence": result.get("confidence", "unknown"),
        }
    except Exception as e:
        return {
            "adr_label": -1, "odr_label": -1, "adr_target": -1, "odr_target": -1, "final_label": -1,
            "label_reason": f"Groq error: {e}", "llm_confidence": "error"
        }

def validate_llm_labels(df: pd.DataFrame) -> None:
    allowed = {-1, 0, 1, 2}
    if "final_label" in df.columns:
        vals = set(df["final_label"].dropna().astype(int).unique().tolist())
        assert vals.issubset(allowed), f"Invalid final_label values: {vals}"

## Load Unlabeled Data and Sample Cases

In [29]:
# Print header
print("=" * 60)
print("LLM-Assisted Labeling (Optional Step)")
print("=" * 60)

# Load unlabeled data
unlabeled_path = OUTPUT_DIR / "needs_llm_labeling.parquet"
if not unlabeled_path.exists():
    print(f"\n[ERROR] {unlabeled_path} not found. Run 04_label_adr.py first.")
    raise FileNotFoundError(f"Required file {unlabeled_path} not found.")

df = pd.read_parquet(unlabeled_path)
print(f"\nUnlabeled cases available: {len(df):,}")

# Sample cases to label
# For a student project, 500–1000 LLM-labeled cases is sufficient as a validation/augmentation set.
# Set SAMPLE_SIZE based on your API budget.
SAMPLE_SIZE = 500
sample = df.sample(min(SAMPLE_SIZE, len(df)), random_state=42).copy()
print(f"Labeling {len(sample)} sampled cases with Gemini API ...")
print("(This will make API calls — check your usage at Google AI Studio)\n")

LLM-Assisted Labeling (Optional Step)

Unlabeled cases available: 2,702,644
Labeling 500 sampled cases with Gemini API ...
(This will make API calls — check your usage at Google AI Studio)



## Label Cases with LLM

In [30]:
# Test Groq API connection
print(f"Testing Groq connection using {GROQ_MODEL}...")
test_prompt = "Respond with a JSON object: {\"status\": \"success\"}"
try:
    test_completion = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": test_prompt}],
        response_format={"type": "json_object"}
    )
    print("✓ Groq API test successful:", test_completion.choices[0].message.content)
except Exception as e:
    print("✗ Groq API test failed:", e)
    raise

Testing Groq connection using llama-3.3-70b-versatile...
✓ Groq API test successful: {
  "status": "success"
}


In [31]:
# Label cases with Groq
results = []
for i, (idx, row) in enumerate(sample.iterrows()):
    # Rotation logic for Groq keys if multiple provided
    if i % 100 == 0 and i > 0 and len(GROQ_API_KEYS) > 1:
        current_key_index = (current_key_index + 1) % len(GROQ_API_KEYS)
        client = get_groq_client()
        print(f"  Switched to Groq key {current_key_index + 1}")

    if i % 50 == 0:
        print(f"  Progress: {i}/{len(sample)} ...")

    result = label_case_with_llm(row)
    results.append(result)
    time.sleep(0.1)  # Groq is fast, but mind rate limits

results_df = pd.DataFrame(results, index=sample.index)
for col in ["adr_label", "odr_label", "adr_target", "odr_target", "final_label", "label_reason", "llm_confidence"]:
    sample[col] = results_df[col]

  Progress: 0/500 ...
  Progress: 50/500 ...
  Progress: 100/500 ...
  Progress: 150/500 ...
  Progress: 200/500 ...
  Progress: 250/500 ...
  Progress: 300/500 ...
  Progress: 350/500 ...
  Progress: 400/500 ...
  Progress: 450/500 ...


## Validate LLM Labels

In [32]:
# Validate the LLM labels
validate_llm_labels(sample)

## Save LLM-Labeled Sample

In [33]:
# Filter out LLM errors and save the sample
sample = sample[sample["final_label"] != -1]  # drop LLM errors

out_path = OUTPUT_DIR / "llm_labeled_sample.parquet"
sample.to_parquet(out_path, index=False)
print(f"\nSaved → {out_path} ({len(sample):,} LLM-labeled rows)")


Saved → /content/drive/MyDrive/MiniProject/compiled_dataset_180426/llm_labeled_sample.parquet (206 LLM-labeled rows)


## Merge LLM Labels into Training Data

In [34]:
# Merge LLM labels back into training_data
training_path = OUTPUT_DIR / "training_data.parquet"
if training_path.exists():
    print("\nMerging LLM labels into training_data.parquet ...")
    train = pd.read_parquet(training_path)
    combined = pd.concat([train, sample], ignore_index=True)
    combined.to_parquet(training_path, index=False)
    print(f"Updated training_data.parquet → {len(combined):,} rows total")


Merging LLM labels into training_data.parquet ...
Updated training_data.parquet → 750,442 rows total


## Display Summary Statistics

In [35]:
# Display summary statistics
print("\n── LLM Label distribution ──────────────────────────────")
label_map = {0: "NOT eligible", 1: "ADR eligible", 2: "ADR+ODR eligible"}
print(sample["final_label"].map(label_map).value_counts())

print("\n── Confidence breakdown ────────────────────────────────")
print(sample["llm_confidence"].value_counts())

print("\nDone ✓")


── LLM Label distribution ──────────────────────────────
final_label
NOT eligible        188
ADR+ODR eligible     18
Name: count, dtype: int64

── Confidence breakdown ────────────────────────────────
llm_confidence
high      130
medium     60
low        16
Name: count, dtype: int64

Done ✓
